<a href="https://colab.research.google.com/github/zxcsheryl/Simhire/blob/main/SimHire_S2_preregistered_analysis_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SimHire Study 2 (coffee store): preregistered analyses

Python version of the Stata do-file (`SimHire_S2_analysis_v3`), limited to the tests in the AsPredicted preregistration (S2_01_AsPredicted):

| Prereg | Test | Section |
|---|---|---|
| Q6 | Exclusions | 1 |
| Q3 | Candidate score, within-person standardization, race gap | 2–3 |
| Q5 | **Primary:** \|race gap\| by condition (regression) + plot | 4 |
| Q8 (a) | Signed race gap by condition | 5 |
| Q8 (b) | Mixed model with a random race slope, slope SD compared between conditions | 6 |
| Q8 (c) | Split-half noise check | 7 |
| Q8 | Exploratory: MRS predicting the race gap, by condition | 8 |

The last cell compares every number with the Stata results. Input: the Qualtrics export with **numeric values** (`..._full_data_value.csv`).

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats, optimize
import statsmodels.formula.api as smf
from matplotlib import pyplot as plt

from google.colab import drive
drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/SimHire_Yu/SimHire_2026_S2_coffee_store/study_design/SimHire_S2_text_raw.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Import and exclusions (prereg Q6)

In [ ]:
# Qualtrics: row 1 = names, rows 2-3 = question text and import IDs
df = pd.read_csv(PATH, skiprows=[1, 2], dtype=str)
print(f"All rows: {len(df)}")

num = lambda s: pd.to_numeric(s, errors="coerce")

# [TEXTFIX] The file is the Qualtrics *choice-text* export, not numeric values.
# Recode the text answers to the numeric codes the rest of the notebook expects.
df["race"] = df["race"].replace({"White": "1"})
df["consent"] = df["consent"].replace({"I understand the information above and agree to participate.": "1"})
df["attn"] = df["attn"].map(lambda a: a if pd.isna(a) else
                            ("1" if a == "Evaluating the shortlisted candidate for each store" else "0"))
mrs_codes = {"Strongly disagree": 1, "Moderately disagree": 2, "Somewhat disagree": 3,
             "Neither agree nor disagree": 4, "Somewhat agree": 5, "Moderately agree": 6, "Strongly agree": 7}
for c in ["mrs_1", "mrs_2", "mrs_3"]:
    assert df[c].dropna().isin(list(mrs_codes)).all(), f"unexpected label in {c}"
    df[c] = df[c].map(mrs_codes)
# [/TEXTFIX]

# Completed = reached the Prolific redirect (FL_25) with ratings recorded.
# Screened-out people also show Finished = True, so Finished alone is not enough.
df = df[(df["Last Seen Flow Element ID"] == "FL_25") & df["rate_1_1"].notna()]
df = df[(num(df["race"]) == 1) & (num(df["consent"]) == 1)]       # White, consented
print(f"Completed: {len(df)}")

# Repeated IP addresses or Prolific IDs: drop ALL copies
df = df[df.groupby("IPAddress")["IPAddress"].transform("size") == 1]
df["PROLIFIC_PID"] = df["PROLIFIC_PID"].fillna("")   # keep blank IDs as "" (as Stata does) so groupby counts them
df = df[df.groupby("PROLIFIC_PID")["PROLIFIC_PID"].transform("size") == 1]
print(f"After duplicates: {len(df)}")

df = df[num(df["attn"]) == 1]                                      # attention check
print(f"Passed attention check: {len(df)}")

# Not preregistered: randomization script failed for this person (no names/photos
# were assigned, race defaulted to "Black" on all 8 pages). Report as an extra exclusion.
df = df[df["ResponseId"] != "R_1flyhlmMRzjlJdv"].reset_index(drop=True)
print(f"Final N: {len(df)}")

df["cond_sim"] = (df["judgment"] == "similarity").astype(int)      # 0 valence, 1 similarity
print(df["cond_sim"].value_counts().rename({0: "valence", 1: "similarity"}))

## 2. Reshape to one row per participant x candidate

The suffix `n` is the display position. `Resume_Store_pair_n` (A–H) is the store/resume shown at position n, `Name_Photo_pair_n` (1–8) the candidate, `rate_n_1..3` the three ratings.

In [ ]:
df["mrs_avg"] = df[["mrs_1", "mrs_2", "mrs_3"]].apply(num).mean(axis=1)   # 1-7, none reverse-worded

long = pd.wide_to_long(
    df[["ResponseId", "cond_sim", "mrs_avg"]
       + [f"{v}_{n}" for v in ["Resume_Store_pair", "Name_Photo_pair", "candidate_race"] for n in range(1, 9)]
       + [f"rate_{n}_{q}" for n in range(1, 9) for q in (1, 2, 3)]]
      .rename(columns={f"rate_{n}_{q}": f"rate{q}_{n}" for n in range(1, 9) for q in (1, 2, 3)}),
    stubnames=["Resume_Store_pair", "Name_Photo_pair", "candidate_race", "rate1", "rate2", "rate3"],
    i="ResponseId", j="n", sep="_").reset_index()

long["store"] = long["Resume_Store_pair"].map({s: i + 1 for i, s in enumerate("ABCDEFGH")})
long["cand"] = num(long["Name_Photo_pair"])
long["black"] = (long["candidate_race"] == "Black").astype(int)
long["blackctr"] = np.where(long["black"] == 1, 1, -1)              # effect coding +1 Black / -1 White
long["score"] = long[["rate1", "rate2", "rate3"]].apply(num).mean(axis=1)   # candidate score, 0-10

# Integrity checks (same as the Stata asserts)
g = long.groupby("ResponseId")
assert (g.size() == 8).all()
assert (g["black"].sum() == 4).all()
assert (g["store"].nunique() == 8).all() and (g["cand"].nunique() == 8).all()
assert (long["black"] == (long["cand"] >= 5)).all()
print(long.shape)   # expect (6488, ...)

(0, 15)


## 3. Within-person standardization and race gap (prereg Q3)

`z` = (score − person's mean of 8) / pooled within-person SD of the condition, where pooled SD = sqrt(Σ dev² / Σ(8 − 1)).
Race gap = mean z of the 4 White − mean z of the 4 Black (+ favors White).

In [ ]:
long["dev"] = long["score"] - long.groupby("ResponseId")["score"].transform("mean")
pooled_sd = long.groupby("cond_sim")["dev"].apply(lambda d: np.sqrt((d**2).sum() / (len(d) - len(d) / 8)))
print("Pooled within-person SD:", pooled_sd.round(6).to_dict())
long["z"] = long["dev"] / long["cond_sim"].map(pooled_sd)

person = (long.pivot_table(index=["ResponseId", "cond_sim", "mrs_avg"], columns="black", values="z", aggfunc="mean")
              .rename(columns={0: "z_white", 1: "z_black"}).reset_index())
person["racegap"] = person["z_white"] - person["z_black"]
person["absracegap"] = person["racegap"].abs()
person.groupby("cond_sim")[["absracegap", "racegap"]].agg(["count", "mean", "std"])

Pooled within-person SD: {}


KeyError: 'z_white'

## 4. Primary test (prereg Q5): is |race gap| smaller under similarity?

Regression with one row per person, robust (HC1) SEs = Stata `reg absracegap i.cond_sim, robust`. The `cond_sim` coefficient is similarity − valence; the prediction is < 0.

In [ ]:
primary = smf.ols("absracegap ~ cond_sim", data=person).fit(cov_type="HC1", use_t=True)
print(primary.summary().tables[1])

# Same comparison as Welch's t-test (Stata: ttest absracegap, by(cond_sim) unequal)
v, s = [person.loc[person.cond_sim == k, "absracegap"] for k in (0, 1)]
print(stats.ttest_ind(v, s, equal_var=False))

In [ ]:
ci = person.groupby("cond_sim")["absracegap"].agg(["mean", "sem", "count"])
ci["half"] = ci["sem"] * stats.t.ppf(0.975, ci["count"] - 1)
plt.bar(["Valence", "Similarity"], ci["mean"], yerr=ci["half"], capsize=6, width=0.5)
plt.ylabel("Mean |race gap| (within-person SD units)")
plt.title("Primary outcome with 95% CI")
plt.show()

## 5. Secondary (a): signed race gap by condition

In [ ]:
v, s = [person.loc[person.cond_sim == k, "racegap"] for k in (0, 1)]
print("Valence vs similarity:", stats.ttest_ind(v, s, equal_var=False))
print("Valence vs 0:         ", stats.ttest_1samp(v, 0))
print("Similarity vs 0:      ", stats.ttest_1samp(s, 0))

## 6. Secondary (b): mixed model with a random race slope

For each person i and candidate j: `z_ij = fixed effects + u_i * blackctr_ij + e_ij`, with Var(u_i) allowed to differ by condition. The SD of u_i is how much race moves each person's ratings, in either direction; the prediction is a smaller SD under similarity. No random intercept, because z is already centered within person (Stata `nocons`).

This is the Stata model `mixed z ... || id: blackctr, nocons`. statsmodels' `MixedLM` can't easily give each condition its own slope variance without a random intercept, so the small function below writes the likelihood out directly. Each person has 8 ratings, so their covariance matrix is σ²I + τ²·x xᵀ (x = the ±1 race codes), which has a simple closed-form inverse and determinant. It reproduces Stata to 4 decimals.

In [ ]:
def fit_random_slope(d, fixed, slope_var_by_condition=True, reml=False):
    # d: long data; fixed: list of fixed-effect columns (intercept added if "const" is listed)
    people = [(grp["z"].to_numpy(), grp["blackctr"].to_numpy(float),
               grp[fixed].to_numpy(float), int(grp["cond_sim"].iloc[0]))
              for _, grp in d.groupby("ResponseId")]
    N, p = 8 * len(people), len(fixed)

    def negloglik(theta):
        s2 = np.exp(theta[0])
        tau2 = np.exp(theta[1:]) if slope_var_by_condition else np.exp([theta[1], theta[1]])
        A = np.zeros((p, p)); b = np.zeros(p); logdet = 0.0; parts = []
        for y, x, F, c in people:
            f = tau2[c] / (s2 + 8 * tau2[c])                # V^-1 = (I - f x x') / s2
            Fx, yx = x @ F, x @ y
            A += (F.T @ F - f * np.outer(Fx, Fx)) / s2
            b += (F.T @ y - f * Fx * yx) / s2
            logdet += 8 * np.log(s2) + np.log(1 + 8 * tau2[c] / s2)
            parts.append((y, x, F, f))
        beta = np.linalg.solve(A, b)
        quad = sum(((y - F @ beta) @ (y - F @ beta) - f * (x @ (y - F @ beta))**2) / s2 for y, x, F, f in parts)
        nll = 0.5 * (logdet + quad + (N - p * reml) * np.log(2 * np.pi))
        if reml:
            nll += 0.5 * np.linalg.slogdet(A)[1]
        return nll, beta

    k = 3 if slope_var_by_condition else 2
    best = min((optimize.minimize(lambda t: negloglik(t)[0], x0, method="L-BFGS-B")
                for x0 in ([-0.2, -2.6, -3.3][:k], [-0.2, -3.3, -2.6][:k], [-0.3, -2.9, -2.9][:k])),
               key=lambda r: r.fun)
    nll, beta = negloglik(best.x)
    tau2 = np.exp(best.x[1:])
    return {"loglik": -nll, "beta": dict(zip(fixed, beta.round(4))), "var_residual": np.exp(best.x[0]),
            "var_slope": tau2, "sd_slope": np.sqrt(tau2)}

long["const"] = 1.0
long["blackXsim"] = long["blackctr"] * long["cond_sim"]

In [ ]:
# (1) Each condition on its own, REML (Stata: mixed z blackctr if cond_sim==k || id: blackctr, nocons reml)
for k, name in [(0, "Valence"), (1, "Similarity")]:
    r = fit_random_slope(long[long.cond_sim == k], ["const", "blackctr"], slope_var_by_condition=False, reml=True)
    print(f"{name:10s}  sd(race slope) = {r['sd_slope'][0]:.4f}   sd(residual) = {np.sqrt(r['var_residual']):.4f}   "
          f"average race effect = {r['beta']['blackctr']:+.4f}")

In [ ]:
# (2) Formal test: separate vs. common slope variance, ML + likelihood-ratio test
#     (Stata: lrtest separate common)
fx = ["const", "blackctr", "cond_sim", "blackXsim"]
sep = fit_random_slope(long, fx, slope_var_by_condition=True)
com = fit_random_slope(long, fx, slope_var_by_condition=False)
LR = 2 * (sep["loglik"] - com["loglik"])
print(f"Separate: var(slope) valence = {sep['var_slope'][0]:.4f}, similarity = {sep['var_slope'][1]:.4f}, "
      f"log likelihood = {sep['loglik']:.4f}")
print(f"Common:   var(slope) = {com['var_slope'][0]:.4f}, log likelihood = {com['loglik']:.4f}")
print(f"LR chi2(1) = {LR:.2f}, p = {stats.chi2.sf(LR, 1):.4f}")

## 7. Secondary (c): split-half noise check

Split each person's 8 candidates into two halves of 2 White + 2 Black (all 6 × 6 = 36 splits). The difference between halves contains no race effect, so its absolute size measures rating noise alone.

In [ ]:
def split_gap(grp):
    grp = grp.sort_values("store")
    w = grp.loc[grp.black == 0, "z"].to_numpy(); b = grp.loc[grp.black == 1, "z"].to_numpy()
    gaps = []
    for wi in combinations(range(4), 2):
        for bi in combinations(range(4), 2):
            wo = [k for k in range(4) if k not in wi]; bo = [k for k in range(4) if k not in bi]
            gaps.append(abs(np.r_[w[list(wi)], b[list(bi)]].mean() - np.r_[w[wo], b[bo]].mean()))
    return np.mean(gaps)

splits = {rid: split_gap(grp) for rid, grp in long.groupby("ResponseId")}
person["splitgap"] = person["ResponseId"].map(splits)
v, s = [person.loc[person.cond_sim == k, "splitgap"] for k in (0, 1)]
print(f"Mean noise-only gap: valence {v.mean():.4f}, similarity {s.mean():.4f}")
print(stats.ttest_ind(v, s, equal_var=False))
print(smf.ols("absracegap ~ cond_sim + splitgap", data=person).fit(cov_type="HC1", use_t=True).summary().tables[1])

## 8. Exploratory: does modern racism (MRS) predict the race gap, by condition?

In [ ]:
person["mrs_ct"] = person["mrs_avg"] - df["mrs_avg"].mean()    # centered, as in Stata (mean over all participants)
print(smf.ols("racegap ~ mrs_ct * cond_sim", data=person).fit(cov_type="HC1", use_t=True).summary().tables[1])
print(smf.ols("absracegap ~ mrs_ct * cond_sim", data=person).fit(cov_type="HC1", use_t=True).summary().tables[1])

## 9. Check against Stata (N = 811, ID 73 excluded)

In [ ]:
check = pd.DataFrame([
    ("Pooled SD valence",            pooled_sd[0],                                1.328034),
    ("Pooled SD similarity",         pooled_sd[1],                                1.322939),
    ("|gap| valence (_cons)",        primary.params["Intercept"],                 0.5592),
    ("|gap| diff (sim - val)",       primary.params["cond_sim"],                  0.0083),
    ("primary p",                    primary.pvalues["cond_sim"],                 0.834),
    ("signed gap valence",           person.query("cond_sim==0").racegap.mean(), -0.0566),
    ("signed gap similarity",        person.query("cond_sim==1").racegap.mean(), -0.1054),
    ("ML var(slope) valence",        sep["var_slope"][0],                         0.0739),
    ("ML var(slope) similarity",     sep["var_slope"][1],                         0.0367),
    ("LR chi2",                      LR,                                          5.69),
    ("split-half valence",           person.query("cond_sim==0").splitgap.mean(), 0.4940),
], columns=["quantity", "python", "stata"])
check["match"] = (check["python"] - check["stata"]).abs() < 0.006
check.round(4)